# 06 Summary numbers

Lesson 12: check every number in the draft insights against the outputs, and build the tables for the summary and the app.

Data: `outputs/*.csv` only (notebooks 01-05). Total impact comes from `scenarios.csv` (Action vs Base), never from adding rows.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs"
BERRY = "MixedBerryPremium"
RULE_ITEMS = ["PassionFruit", BERRY]
PLAN_VARIANT = "CR+0.1, promo x lift"

read = lambda name: pd.read_csv(OUT / f"{name}.csv")
company, kitchen, variance = read("company_pnl"), read("kitchen_pnl"), read("budget_variance")
ue, price, promo = read("unit_economics"), read("price_recommendation").set_index("sku"), read("promo_incremental")
rule_bt, levers, scen = read("prep_rule_backtest"), read("scenario_levers").set_index("step"), read("scenarios")
plan, portfolio = read("profit_plan"), read("portfolio_decision")

## 1. Are the draft numbers right?

In [2]:
checks = []


def check(claim, draft, value, source, ok):
    """One line of the claim check. `ok` = the draft number matches the data after rounding."""
    checks.append({"claim": claim, "draft": draft, "checked": value, "source": source, "ok": bool(ok)})


# 1. Budget and overhead
gp_var = variance["gp_var"].sum()
op = company["operating_profit"].sum()
kop = kitchen.groupby("kitchen")["operating_profit"].sum()
bkk, pattaya = kop.filter(like="BKK").sum(), kop.filter(like="Pattaya").sum()
check(1, "misses budget", f"GP-C {gp_var / 1000:,.0f}k vs budget ({gp_var / variance['budget_gp'].sum() * 100:.1f}%)",
      "budget_variance.csv", gp_var < 0)
check(1, "loss-making after overhead", f"operating profit {op / 1000:,.0f}k", "company_pnl.csv", op < 0)
check(1, "Bangkok kitchens drive it", f"BKK {bkk / 1e6:.2f}M vs Pattaya {pattaya / 1e6:+.2f}M after overhead",
      "kitchen_pnl.csv", bkk < 0 < pattaya)

# 2. Berry
b = ue[(ue["level"] == "item") & (ue["sku"] == BERRY)].iloc[0]
check(2, "-1.92 THB a cup", f"{b['cm_after_waste_per_cup']:.2f}", "unit_economics.csv (Mar-Aug)",
      round(b["cm_after_waste_per_cup"], 2) == -1.92)
check(2, "20.5% thrown away", f"{b['waste_rate_pct']:.1f}%", "unit_economics.csv", round(b["waste_rate_pct"], 1) == 20.5)
check(2, "-43k a month at August volume", f"{-price.loc[BERRY, 'delist_thb'] / 1000:.1f}k", "price_recommendation.csv",
      round(price.loc[BERRY, "delist_thb"] / 1000) == 43)

# 3. Promos
code_rows = promo[promo["level"] == "code"].set_index("code")
net_month = code_rows["net"].sum() / 12
check(3, "-14.6k THB a month after waste", f"{net_month / 1000:.1f}k", "promo_incremental.csv (3 codes)",
      round(net_month / 1000, 1) == -14.6)
# Waste cups on RC102 days: printed in notebook 03, section on waste (2,045 vs 1,131 on comparable days).
rc102_waste = 2045 / 1131 - 1
rc102_lift = code_rows.loc["RC102", "actual_lift_pct"] / 100
check(3, "+81% waste on RC102", f"+{rc102_waste:.0%} waste cups; sales +{rc102_lift:.0%}",
      "notebook 03 output; promo_incremental.csv", round(rc102_waste * 100) == 81)
check(3, "no promo pays back", f"all 3 codes below 0: {(code_rows['net'] < 0).all()}", "promo_incremental.csv",
      (code_rows["net"] < 0).all())

# 4. Prices
for sku in ["Watermelon", "PassionFruit"]:
    r = price.loc[sku]
    check(4, f"{sku}: break-even far below measured",
          f"break-even {r['breakeven_elasticity']:.2f} vs {r['elasticity']:.2f} (CI {r['ci_low']:.2f} to {r['ci_high']:.2f}, "
          f"day-level {r['day_elasticity']:.2f}); gap to CI low {r['ci_low'] - r['breakeven_elasticity']:.2f}",
          "price_recommendation.csv", r["breakeven_elasticity"] < r["ci_low"])

# 5. Waste and the prep rule
waste_fy = company["waste_cost"].sum()
aug = rule_bt[(rule_bt["period"] == "2026-08") & (rule_bt["variant"] == PLAN_VARIANT) & rule_bt["sku"].isin(RULE_ITEMS)]
model_rule = levers.loc["Prep rule (berry, PassionFruit)", "elasticity_-1.5"]
check(5, "waste 860k a year", f"{waste_fy / 1000:.1f}k", "company_pnl.csv", round(waste_fy / 1000) == 860)
check(5, "prep rule saves X a month",
      f"{aug['saving_per_month'].sum() / 1000:.1f}k (Aug backtest) to {model_rule / 1000:.1f}k (Sep-Nov model)",
      "prep_rule_backtest.csv; scenario_levers.csv", True)

checks = pd.DataFrame(checks)
pd.set_option("display.max_colwidth", 120)
display(checks)
assert checks["ok"].all(), checks[~checks["ok"]]

,claim,draft,checked,source,ok
0,1,misses budget,GP-C -764k vs budget (-9.1%),budget_variance.csv,True
1,1,loss-making after overhead,operating profit -404k,company_pnl.csv,True
2,1,Bangkok kitchens drive it,BKK -1.11M vs Pattaya +0.70M after overhead,kitchen_pnl.csv,True
3,2,-1.92 THB a cup,-1.92,unit_economics.csv (Mar-Aug),True
4,2,20.5% thrown away,20.5%,unit_economics.csv,True
5,2,-43k a month at August volume,-43.3k,price_recommendation.csv,True
6,3,-14.6k THB a month after waste,-14.6k,promo_incremental.csv (3 codes),True
7,3,+81% waste on RC102,+81% waste cups; sales +75%,notebook 03 output; promo_incremental.csv,True
8,3,no promo pays back,all 3 codes below 0: True,promo_incremental.csv,True
9,4,Watermelon: break-even far below measured,"break-even -2.24 vs -1.66 (CI -1.91 to -1.40, day-level -1.81); gap to CI low 0.32",price_recommendation.csv,True


**All 13 draft numbers match the data.** Three claims are worded stronger than the data, so the titles below are adjusted:
the Bangkok kitchens drive the loss after overhead, not the budget miss (berry and PassionFruit margin gaps -1.1M; Pattaya is also below the budget margin).
RC102 wastes +81% cups but sells +75%, so the waste rate does not rise. Watermelon's break-even (-2.24) is 0.32 below the CI low: headroom, not far.
X for the prep rule = 29.6k (August backtest) to 40.2k (Sep-Nov model).

## 2. What is the total impact a month? (Sep-Nov 2026)

In [3]:
gp = scen.groupby(["scenario", "month"])["gross_profit"].sum().unstack("month")
op_m = gp - 670_000                      # overhead is the same in every scenario
delta = (op_m.loc["Action"] - op_m.loc["Base"])
delta_low_eps = (op_m.loc["Action (elasticity -0.5)"] - op_m.loc["Base"])
total = pd.DataFrame({"elasticity -1.5": delta, "elasticity -0.5": delta_low_eps})
total.loc["mean a month"] = total.mean()
display((total / 1000).round(1))

# The lever bridge (same model, one lever at a time) must add up to the same total.
lv = levers.loc[["Kill RC101 W/PF/berry + RC103", "Watermelon 49, PassionFruit 69", "Berry 99",
                 "Prep rule (berry, PassionFruit)"]]
assert np.isclose(lv["elasticity_-1.5"].sum(), total.loc["mean a month", "elasticity -1.5"])
assert np.isclose(lv["elasticity_-0.5"].sum(), total.loc["mean a month", "elasticity -0.5"])
TOTAL_LOW, TOTAL_HIGH = sorted(total.loc["mean a month"])
print(f"Total impact: +{TOTAL_LOW / 1000:.1f}k to +{TOTAL_HIGH / 1000:.1f}k THB a month (Action vs Base, Sep-Nov 2026)")

outlook = op_m.loc[["Base", "Action", "Action + delist berry", "Downside", "Base downside"]]
outlook["3 months"] = outlook.sum(axis=1)
display((outlook / 1000).round(0))

,elasticity -1.5,elasticity -0.5
month,,
2026-09,94.4,118.4
2026-10,96.6,120.9
2026-11,86.3,115.1
mean a month,92.4,118.1


Total impact: +92.4k to +118.1k THB a month (Action vs Base, Sep-Nov 2026)


month,2026-09,2026-10,2026-11,3 months
scenario,,,,
Base,-267.0,-257.0,-161.0,-685.0
Action,-173.0,-160.0,-75.0,-408.0
Action + delist berry,-186.0,-175.0,-89.0,-450.0
Downside,-291.0,-282.0,-214.0,-787.0
Base downside,-399.0,-393.0,-315.0,-1106.0


**Action vs Base: +92.4k (elasticity -1.5) to +118.1k (-0.5) THB a month.** Same as the lever bridge (asserted).
Still below 0 in every month: -408k in 3 months vs -685k for Base; Downside -787k vs -1,106k for Base under the same shock.

## 3. What do we recommend, and who owns it?

In [4]:
# Tests: lesson-9 upside of the Pineapple and Guava price tests (RC101 modify inside) + RC102 at 12%.
tests_high = (portfolio.set_index("name").loc[["Pineapple", "Guava"], "thb_month_high"].sum()
              + read("promo_decision").query("code == 'RC102'")["modify_net"].sum())
close_high = portfolio.set_index("name").loc["BKK_Ladprao close (review only)", "thb_month_high"]


def lev(step):
    """Low and high THB a month of one lever, across the two elasticities."""
    r = levers.loc[step]
    return min(r["elasticity_-1.5"], r["elasticity_-0.5"]), max(r["elasticity_-1.5"], r["elasticity_-0.5"])


rows = [
    ("Waste", plan.loc[0, "action"], *lev("Prep rule (berry, PassionFruit)"), True, "medium",
     "No sold-out data, so today's prep looks better than it is; berry runs out on ~17% of demand by design",
     plan.loc[0, "owner"], 0),
    ("Pricing", plan.loc[1, "action"], *lev("Watermelon 49, PassionFruit 69"), True, "medium",
     "Elasticity -1.5 (low) to -0.5 (high); measured -1.66 and -1.24 from 7-8 promo days",
     plan.loc[1, "owner"], 1),
    ("Menu", plan.loc[2, "action"], *lev("Berry 99"), True, "low",
     "Berry fruit at the Jun-Aug mean (45.1 THB); at the late-Aug 51.2 THB berry earns ~11k a month less",
     plan.loc[2, "owner"], 2),
    ("Promotions", plan.loc[3, "action"], *lev("Kill RC101 W/PF/berry + RC103"), True, "high",
     "Killed promo cups fall back to the lesson-7 baseline; spillover and pull-forward left out",
     plan.loc[3, "owner"], 3),
    ("Tests", plan.loc[4, "action"], 0.0, tests_high, False, "low",
     "Upside only if tests pass; Mar-Aug base (April peak inside), so Sep-Nov values are smaller",
     plan.loc[4, "owner"], 4),
    ("Kitchens", plan.loc[5, "action"], 0.0, close_high, False, "low",
     "High = close Ladprao with all overhead avoidable and no customer moving; lease terms unknown",
     plan.loc[5, "owner"], 5),
]
rec = pd.DataFrame(rows, columns=["area", "action", "thb_month_low", "thb_month_high", "in_total", "confidence",
                                  "key_assumption", "owner", "plan_row"])
rec = rec.join(plan[["start_date", "kpi", "stop_rule"]], on="plan_row").drop(columns="plan_row")
rec[["thb_month_low", "thb_month_high"]] = rec[["thb_month_low", "thb_month_high"]].round(-2)
total_row = {"area": "Total (Action vs Base, from scenarios.csv)", "action": "Rows 1-4 together in one model",
             "thb_month_low": round(TOTAL_LOW, -2), "thb_month_high": round(TOTAL_HIGH, -2), "in_total": True,
             "confidence": "medium", "key_assumption": "Not the sum of rows: each row's low and high can come from a "
             "different elasticity", "owner": "COO", "start_date": "2026-09-01", "kpi": "Operating profit a month",
             "stop_rule": ""}
rec = pd.concat([rec, pd.DataFrame([total_row])], ignore_index=True)
rec.to_csv(OUT / "recommendations.csv", index=False)
display(rec[["area", "thb_month_low", "thb_month_high", "in_total", "confidence", "owner"]])

,area,thb_month_low,thb_month_high,in_total,confidence,owner
0,Waste,38800.0,40200.0,True,medium,"Kitchen managers (4), ops lead checks weekly"
1,Pricing,27200.0,51900.0,True,medium,Revenue manager
2,Menu,13500.0,15900.0,True,low,Menu owner
3,Promotions,11500.0,11500.0,True,high,Marketing / platform manager
4,Tests,0.0,77600.0,False,low,Revenue analyst
5,Kitchens,0.0,51700.0,False,low,Finance / COO
6,"Total (Action vs Base, from scenarios.csv)",92400.0,118100.0,True,medium,COO


## 4. Which insights go in the summary?

In [5]:
insights = pd.DataFrame([
    (1, "The company misses budget and is loss-making after overhead; the Bangkok kitchens drive the loss.",
     f"GP-C {company['gross_profit'].sum() / 1e6:.2f}M vs 8.4M budget ({gp_var / 1000:,.0f}k); operating profit "
     f"{op / 1000:,.0f}k; Bangkok {bkk / 1e6:.2f}M vs Pattaya {pattaya / 1e6:+.2f}M after overhead.",
     "Notebook 04, section 6: operating profit per kitchen and month (heatmap)", "high"),
    (2, "Mixed Berry loses money after waste: -1.92 THB a cup, 20.5% thrown away, -43k a month at August volume.",
     f"CM {b['cm_per_cup']:.2f} THB a cup before waste, waste {b['waste_per_cup']:.2f} THB a cup (Mar-Aug); "
     "39% of all waste THB with 4.5% of cups.",
     "Notebook 04, section 7: menu matrix; notebook 05, section 1: waste by item", "high"),
    (3, "No promo code pays back: -14.6k THB a month after waste; promo days waste more cups (+81% on RC102), "
        "in line with the extra sales.",
     f"RC101, RC102 and RC103 all lose CM after waste and side effects; RC102 sells +{rc102_lift:.0%} and wastes "
     f"+{rc102_waste:.0%} cups, so the waste rate stays about the same.",
     "Notebook 03, section 10: which promos earn back their cost", "high (kill) / low (tests)"),
    (4, "Price headroom on Watermelon and PassionFruit: break-even elasticity is below what we measured "
        "(far below for PassionFruit).",
     "Watermelon break-even -2.24 vs measured -1.66 (day-level -1.81); PassionFruit -3.50 vs -1.24.",
     "Notebook 02, section 2: price simulation (+5 THB at elasticity -0.5 to -1.5)", "medium"),
    (5, "Waste is 860k a year; the prep rule for berry and PassionFruit saves about 30-40k a month.",
     f"+{aug['saving_per_month'].sum() / 1000:.1f}k in the August backtest, +{model_rule / 1000:.1f}k in the Sep-Nov model; "
     "it loses for the 3 cheap items, and no sold-out data exists.",
     "Notebook 05, section 3: today vs rule by month", "medium"),
], columns=["rank", "title", "evidence", "chart", "confidence"])
insights.to_csv(OUT / "insights.csv", index=False)
display(insights)

,rank,title,evidence,chart,confidence
0,1,The company misses budget and is loss-making after overhead; the Bangkok kitchens drive the loss.,GP-C 7.64M vs 8.4M budget (-764k); operating profit -404k; Bangkok -1.11M vs Pattaya +0.70M after overhead.,"Notebook 04, section 6: operating profit per kitchen and month (heatmap)",high
1,2,"Mixed Berry loses money after waste: -1.92 THB a cup, 20.5% thrown away, -43k a month at August volume.","CM 10.02 THB a cup before waste, waste 11.94 THB a cup (Mar-Aug); 39% of all waste THB with 4.5% of cups.","Notebook 04, section 7: menu matrix; notebook 05, section 1: waste by item",high
2,3,"No promo code pays back: -14.6k THB a month after waste; promo days waste more cups (+81% on RC102), in line with th...","RC101, RC102 and RC103 all lose CM after waste and side effects; RC102 sells +75% and wastes +81% cups, so the waste...","Notebook 03, section 10: which promos earn back their cost",high (kill) / low (tests)
3,4,Price headroom on Watermelon and PassionFruit: break-even elasticity is below what we measured (far below for Passio...,Watermelon break-even -2.24 vs measured -1.66 (day-level -1.81); PassionFruit -3.50 vs -1.24.,"Notebook 02, section 2: price simulation (+5 THB at elasticity -0.5 to -1.5)",medium
4,5,Waste is 860k a year; the prep rule for berry and PassionFruit saves about 30-40k a month.,"+29.6k in the August backtest, +40.2k in the Sep-Nov model; it loses for the 3 cheap items, and no sold-out data exi...","Notebook 05, section 3: today vs rule by month",medium


## Findings -> next steps
1. All 13 draft numbers check out; 3 claims reworded to fit the data -> use the reworded titles in the summary and the app.
2. Action adds +92k to +118k THB a month in Sep-Nov (scenarios.csv, not a sum of rows) -> start the 4 in-model levers on 2026-09-01.
3. Sep-Nov still loses 408k with Action -> BKK overhead review is the next lever (Ladprao close: up to +51.7k a month, lease terms unknown).
4. Tests could add up to 77.6k a month on the Mar-Aug base -> run them, keep them out of the plan until they pass.